# 04 - EfficientNet Deep Embedding Extraction

This notebook reads saved ECG scalogram images and extracts deep CNN embeddings.

Recommended flow:

1. Start with EfficientNet-B0 to prove the pipeline.
2. Switch to EfficientNet-B4 for the final feature extraction run.
3. Cache embeddings to avoid recomputing them.
4. Export a feature table that can be merged with handcrafted ECG features.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

PROJECT_DIR = Path("..").resolve()
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(SRC_DIR))

from config import CONFIG
from reproducibility import set_global_seed

seed_state = set_global_seed(CONFIG.seed)
MODEL_NAME = CONFIG.active_model_name
MAX_IMAGES = CONFIG.active_max_records
MANIFEST_PATH = CONFIG.scalogram_manifest_path(MODEL_NAME)
EMBEDDING_DIR = CONFIG.embedding_dir / MODEL_NAME
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH).head(MAX_IMAGES).copy()
print("Images available:", len(manifest))
print("Seed state:", seed_state)
display(manifest.head())


Images available: 1000
Seed state: {'seed': 42, 'python': True, 'numpy': True, 'torch': True, 'cuda': False}


,record_id,mat_path,hea_path,image_path,label,label_name,age,sex,dx_codes,sampling_frequency,n_leads,n_samples,lead_indices,lead_names,image_height,image_width
0,JS00001,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1,Arrhythmia,85,1,"164889003,59118001,164934002",500,12,5000,"0,1,10","I,II,V5",380,380
1,JS00002,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1,Arrhythmia,59,0,"426177001,164934002",500,12,5000,"0,1,10","I,II,V5",380,380
2,JS00004,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1,Arrhythmia,66,1,426177001,500,12,5000,"0,1,10","I,II,V5",380,380
3,JS00005,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1,Arrhythmia,73,0,"164890007,429622005,428750005",500,12,5000,"0,1,10","I,II,V5",380,380
4,JS00006,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\Jennie mams thi...,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1,Arrhythmia,46,0,426177001,500,12,5000,"0,1,10","I,II,V5",380,380


## Dependency Check

This checks whether the deep-learning packages are installed in the active Jupyter kernel. Do not install them on every run; install once, restart the kernel, then continue.


In [2]:
import importlib.util

missing = [pkg for pkg in ["torch", "torchvision"] if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages:", missing)
    print("Run this once, then restart the kernel:")
    print(f"{sys.executable} -m pip install torch torchvision timm")
else:
    print("Deep learning dependencies are available.")


Deep learning dependencies are available.


## Load EfficientNet

This notebook uses `torchvision` first because it is simpler. If you want EfficientNet-B4, install `torch` and `torchvision`, then switch the model name in the cell below.

For quick testing:

- `efficientnet_b0`

For final advisor-requested run:

- `efficientnet_b4`


In [3]:
import torch
from torch import nn
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 if DEVICE == "cpu" else 32
EXPECTED_EMBEDDING_DIM = CONFIG.model_embedding_dim(MODEL_NAME)

if DEVICE == "cpu":
    torch.set_num_threads(4)

if MODEL_NAME == "efficientnet_b0":
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
elif MODEL_NAME == "efficientnet_b4":
    weights = models.EfficientNet_B4_Weights.DEFAULT
    model = models.efficientnet_b4(weights=weights)
else:
    raise ValueError(f"Unsupported model: {MODEL_NAME}")

transform = weights.transforms()

# Remove classifier; output is the pooled CNN embedding.
model.classifier = nn.Identity()
model.eval().to(DEVICE)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Batch size:", BATCH_SIZE)
print("Max images this run:", MAX_IMAGES)
print("Expected embedding dimension:", EXPECTED_EMBEDDING_DIM)
print("Transform:", transform)


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to C:\Users\Admin/.cache\torch\hub\checkpoints\efficientnet_b4_rwightman-23ab8bcd.pth


  0%|          | 0.00/74.5M [00:00<?, ?B/s]

 10%|█         | 7.50M/74.5M [00:00<00:00, 78.4MB/s]

 20%|██        | 15.0M/74.5M [00:00<00:00, 75.8MB/s]

 33%|███▎      | 24.4M/74.5M [00:00<00:00, 85.9MB/s]

...

 70%|███████   | 52.5M/74.5M [00:00<00:00, 94.0MB/s]

 83%|████████▎ | 61.6M/74.5M [00:00<00:00, 94.1MB/s]

 95%|█████████▍| 70.8M/74.5M [00:00<00:00, 94.4MB/s]

100%|██████████| 74.5M/74.5M [00:00<00:00, 91.3MB/s]

Device: cpu
Model: efficientnet_b4
Batch size: 8
Max images this run: 1000
Expected embedding dimension: 1792
Transform: ImageClassification(
    crop_size=[380]
    resize_size=[384]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)


## Extract Or Load Cached Embeddings

Each image gets a cached `.npy` embedding. Re-running the notebook will reuse existing embeddings.


In [4]:
def load_image_tensor(image_path):
    image = Image.open(image_path).convert("RGB")
    return transform(image)


def embedding_path(record_id):
    return EMBEDDING_DIR / f"{record_id}_{MODEL_NAME}.npy"


work_manifest = manifest.copy()
rows = []
cached_count = 0
new_count = 0

for start in tqdm(range(0, len(work_manifest), BATCH_SIZE), desc="Extracting embeddings"):
    batch_df = work_manifest.iloc[start:start + BATCH_SIZE]
    tensors = []
    uncached_rows = []

    for _, row in batch_df.iterrows():
        cache_path = embedding_path(row["record_id"])
        if cache_path.exists():
            embedding = np.load(cache_path)
            rows.append({
                "record_id": row["record_id"],
                "label": row["label"],
                "label_name": row["label_name"],
                "embedding_path": str(cache_path),
                "embedding_dim": int(embedding.shape[-1]),
            })
            cached_count += 1
        else:
            tensors.append(load_image_tensor(row["image_path"]))
            uncached_rows.append(row)

    if tensors:
        batch = torch.stack(tensors).to(DEVICE)
        with torch.no_grad():
            embeddings = model(batch).detach().cpu().numpy()

        for row, embedding in zip(uncached_rows, embeddings):
            if embedding.shape[-1] != EXPECTED_EMBEDDING_DIM:
                raise ValueError(
                    f"{MODEL_NAME} produced {embedding.shape[-1]} features; "
                    f"expected {EXPECTED_EMBEDDING_DIM}."
                )
            cache_path = embedding_path(row["record_id"])
            np.save(cache_path, embedding)
            rows.append({
                "record_id": row["record_id"],
                "label": row["label"],
                "label_name": row["label_name"],
                "embedding_path": str(cache_path),
                "embedding_dim": int(embedding.shape[-1]),
            })
            new_count += 1

embedding_manifest = pd.DataFrame(rows)
embedding_manifest_path = CONFIG.embedding_manifest_path(MODEL_NAME)
embedding_manifest.to_csv(embedding_manifest_path, index=False)

print(f"Saved embedding manifest: {embedding_manifest_path}")
print(f"Loaded from cache: {cached_count:,}")
print(f"Newly extracted: {new_count:,}")
display(embedding_manifest.head())


Extracting embeddings:   0%|          | 0/125 [00:00<?, ?it/s]

Saved embedding manifest: C:\Users\Admin\Desktop\Kshitiz\healthcare project\healthcare project\outputs\deep_features\manifests\efficientnet_b4_embedding_manifest.csv
Loaded from cache: 0
Newly extracted: 1,000


,record_id,label,label_name,embedding_path,embedding_dim
0,JS00001,1,Arrhythmia,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1792
1,JS00002,1,Arrhythmia,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1792
2,JS00004,1,Arrhythmia,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1792
3,JS00005,1,Arrhythmia,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1792
4,JS00006,1,Arrhythmia,C:\Users\Admin\Desktop\Kshitiz\healthcare proj...,1792


## Build Deep Feature Table

This converts cached embeddings into a flat CSV table:

`record_id, label, label_name, eff_0000, eff_0001, ...`


In [5]:
embedding_matrix = np.vstack([
    np.load(path).ravel()
    for path in tqdm(embedding_manifest["embedding_path"], desc="Loading cached embeddings")
])
if embedding_matrix.shape[1] != EXPECTED_EMBEDDING_DIM:
    raise ValueError(
        f"Deep matrix has {embedding_matrix.shape[1]} columns; "
        f"expected {EXPECTED_EMBEDDING_DIM}."
    )

deep_columns = [f"eff_{i:04d}" for i in range(EXPECTED_EMBEDDING_DIM)]
deep_feature_table = pd.concat(
    [
        embedding_manifest[["record_id", "label", "label_name"]].reset_index(drop=True),
        pd.DataFrame(embedding_matrix, columns=deep_columns),
    ],
    axis=1,
)
deep_feature_path = CONFIG.deep_feature_path(MODEL_NAME)
deep_feature_table.to_csv(deep_feature_path, index=False)

print("Deep feature table shape:", deep_feature_table.shape)
print(f"Saved: {deep_feature_path}")
display(deep_feature_table.head())


Loading cached embeddings:   0%|          | 0/1000 [00:00<?, ?it/s]

Deep feature table shape: (1000, 1795)
Saved: C:\Users\Admin\Desktop\Kshitiz\healthcare project\healthcare project\outputs\deep_features\features\efficientnet_b4_deep_features.csv


,record_id,label,label_name,eff_0000,eff_0001,eff_0002,eff_0003,eff_0004,eff_0005,eff_0006,...,eff_1782,eff_1783,eff_1784,eff_1785,eff_1786,eff_1787,eff_1788,eff_1789,eff_1790,eff_1791
0,JS00001,1,Arrhythmia,-0.123473,-0.097806,-0.123424,-0.081698,0.250855,-0.108303,-0.113441,...,-0.047898,-0.090787,-0.113593,-0.054016,-0.001269,-0.096003,0.038682,-0.091391,-0.033740,-0.127014
1,JS00002,1,Arrhythmia,-0.129252,-0.104707,-0.129405,-0.127796,-0.034627,-0.052273,-0.121716,...,-0.045695,-0.140099,-0.138972,-0.067695,-0.092805,-0.072985,-0.061193,-0.097133,-0.055327,-0.123373
2,JS00004,1,Arrhythmia,-0.138780,-0.097281,-0.109482,-0.122959,-0.143758,-0.083260,-0.119552,...,-0.058728,-0.138212,-0.091003,-0.073740,-0.089393,-0.112053,-0.043016,-0.094094,-0.064179,-0.129731
3,JS00005,1,Arrhythmia,-0.126134,-0.082237,-0.123019,-0.082124,0.168547,-0.069919,-0.077190,...,-0.032222,-0.146288,-0.112166,-0.064593,-0.059529,-0.121557,-0.047416,-0.081791,-0.063348,-0.130503
4,JS00006,1,Arrhythmia,-0.108500,-0.098391,-0.121231,-0.115688,-0.084316,-0.078543,-0.137710,...,-0.034169,-0.151968,-0.109000,-0.096042,-0.087323,-0.105370,-0.086498,-0.074867,-0.062081,-0.092885


## Optional: Merge With Handcrafted Features

This prepares the hybrid table. It expects the existing handcrafted CSV to contain `filename` values matching `record_id`.


In [6]:
HANDCRAFTED_PATH = CONFIG.feature_dir / "enhanced_handcrafted_features.csv"
handcrafted = pd.read_csv(HANDCRAFTED_PATH)

hybrid = handcrafted.merge(
    deep_feature_table.drop(columns=["label_name"]),
    on=["record_id", "label"],
    how="inner",
)

hybrid_path = CONFIG.hybrid_feature_path(MODEL_NAME)
hybrid.to_csv(hybrid_path, index=False)

print("Handcrafted shape:", handcrafted.shape)
print("Deep shape:", deep_feature_table.shape)
print("Hybrid shape:", hybrid.shape)
print(f"Saved: {hybrid_path}")
assert len(hybrid) == len(deep_feature_table), "Hybrid merge dropped records."
assert len([c for c in hybrid.columns if c.startswith("eff_")]) == EXPECTED_EMBEDDING_DIM
display(hybrid.head())


Handcrafted shape: (1000, 22)
Deep shape: (1000, 1795)
Hybrid shape: (1000, 1814)
Saved: C:\Users\Admin\Desktop\Kshitiz\healthcare project\healthcare project\outputs\deep_features\features\hybrid_handcrafted_efficientnet_b4.csv


,record_id,label,label_name,age,sex,mean,std,max,min,energy,...,eff_1782,eff_1783,eff_1784,eff_1785,eff_1786,eff_1787,eff_1788,eff_1789,eff_1790,eff_1791
0,JS00001,1,Arrhythmia,85,1,3.051758e-09,1.0,4.616830,-5.410841,5000.000000,...,-0.047898,-0.090787,-0.113593,-0.054016,-0.001269,-0.096003,0.038682,-0.091391,-0.033740,-0.127014
1,JS00002,1,Arrhythmia,59,0,7.629394e-10,1.0,7.913949,-1.121108,5000.000000,...,-0.045695,-0.140099,-0.138972,-0.067695,-0.092805,-0.072985,-0.061193,-0.097133,-0.055327,-0.123373
2,JS00004,1,Arrhythmia,66,1,1.068115e-08,1.0,6.939423,-0.814498,5000.000488,...,-0.058728,-0.138212,-0.091003,-0.073740,-0.089393,-0.112053,-0.043016,-0.094094,-0.064179,-0.129731
3,JS00005,1,Arrhythmia,73,0,3.814697e-09,1.0,4.638732,-2.487807,5000.000977,...,-0.032222,-0.146288,-0.112166,-0.064593,-0.059529,-0.121557,-0.047416,-0.081791,-0.063348,-0.130503
4,JS00006,1,Arrhythmia,46,0,3.814697e-09,1.0,6.781251,-1.004105,5000.000000,...,-0.034169,-0.151968,-0.109000,-0.096042,-0.087323,-0.105370,-0.086498,-0.074867,-0.062081,-0.092885
